# 05 — V1 Dataset Validation & Ground-Truth Audit

Notebook 04 is complete. This notebook **does not train the PINN**.

It validates the 7.6M-row PyBaMM dataset before model training:

- artifact existence and hashes
- schema and units
- scenario integrity
- exact 100-cell coverage
- Cell 37 and non-37 fault coverage
- fault-location leakage
- instance-level train/validation/test split
- healthy/degraded separability
- time ordering
- future-target definition without leakage
- particle profiles
- pack data
- external-data compatibility

**GPU:** CPU is sufficient. Do not connect a T4 for this notebook.

**Critical:** `cell_timeseries.csv` is large, so every operation on it is chunked. Never use `pd.read_csv()` on the entire file.


In [2]:

from google.colab import drive
drive.mount("/content/drive", force_remount=False)

from pathlib import Path
import json, hashlib, re, random
import numpy as np
import pandas as pd

SEED = 20260927
random.seed(SEED)
np.random.seed(SEED)

MYDRIVE = Path("/content/drive/MyDrive")
V1_DIR = MYDRIVE / "PINN_EV_Battery" / "data" / "v1"

if not V1_DIR.exists():
    matches = list(MYDRIVE.rglob("cell_timeseries.csv"))
    dirs = sorted({p.parent for p in matches if p.parent.name == "v1"})
    if len(dirs) == 1:
        V1_DIR = dirs[0]
    elif not dirs:
        raise FileNotFoundError("Cannot find PINN_EV_Battery/data/v1/cell_timeseries.csv on Drive.")
    else:
        raise RuntimeError("Multiple v1 directories found. Set V1_DIR manually:\n" + "\n".join(map(str, dirs)))

print("V1_DIR:", V1_DIR)


Mounted at /content/drive
V1_DIR: /content/drive/MyDrive/PINN_EV_Battery/data/v1


In [ ]:
# ============================================================
# NOTEBOOK 05 — CELL 1
# V1 ARTIFACT DISCOVERY + PROVENANCE REPAIR
# ============================================================

from pathlib import Path
import json
import hashlib
import pandas as pd

print("=" * 80)
print("NOTEBOOK 05 — V1 ARTIFACT DISCOVERY")
print("=" * 80)

# ------------------------------------------------------------
# 1. Project root
# ------------------------------------------------------------

DRIVE_ROOT = Path("/content/drive/MyDrive/PINN_EV_Battery")

if not DRIVE_ROOT.exists():
    raise FileNotFoundError(
        f"STOP — project directory not found:\n{DRIVE_ROOT}\n"
        "Mount Google Drive before continuing."
    )

V1_DIR = DRIVE_ROOT / "data" / "v1"

if not V1_DIR.exists():
    raise FileNotFoundError(
        f"STOP — V1 data directory not found:\n{V1_DIR}"
    )

print("Project root:", DRIVE_ROOT)
print("V1 directory:", V1_DIR)


# ------------------------------------------------------------
# 2. Actual Notebook 04 outputs
# ------------------------------------------------------------

ARTIFACTS = {
    "cell_timeseries":
        V1_DIR / "cell_timeseries.csv",

    "particle_profiles":
        V1_DIR / "particle_profiles.csv",

    "pack_timeseries":
        V1_DIR / "pack_timeseries.csv",

    "scenario_instances":
        V1_DIR / "scenario_instances.csv",

    "s6_intervention_summary":
        V1_DIR / "s6_intervention_summary.csv",

    "generation_metadata":
        V1_DIR / "v1_generation_metadata.json",
}


# ------------------------------------------------------------
# 3. Verify the real generated files
# ------------------------------------------------------------

missing = [
    name
    for name, path in ARTIFACTS.items()
    if not path.exists()
]

if missing:
    raise FileNotFoundError(
        "STOP — required Notebook 04 artifacts are missing:\n"
        + "\n".join(
            f"  - {name}: {ARTIFACTS[name]}"
            for name in missing
        )
    )

print()
print("Notebook 04 artifacts:")
for name, path in ARTIFACTS.items():
    size_mb = path.stat().st_size / (1024 ** 2)
    print(
        f"  PASS: {name:25s} "
        f"{size_mb:,.2f} MB"
    )


# ------------------------------------------------------------
# 4. Create data dictionary from ACTUAL files
# ------------------------------------------------------------
#
# This is not fabricated scientific metadata.
# It records the schemas actually present in the files.
#

DATA_DICTIONARY_PATH = (
    V1_DIR / "v1_data_dictionary.json"
)

data_dictionary = {
    "source": "Notebook 04 generated artifacts",
    "generated_by": "Notebook 05 provenance repair",
    "files": {}
}


for name in [
    "cell_timeseries",
    "particle_profiles",
    "pack_timeseries",
    "scenario_instances",
    "s6_intervention_summary",
]:

    path = ARTIFACTS[name]

    # Read only the header.
    df_head = pd.read_csv(
        path,
        nrows=5
    )

    columns = []

    for col in df_head.columns:

        dtype = str(
            df_head[col].dtype
        )

        columns.append({
            "name": str(col),
            "dtype_sample": dtype
        })

    data_dictionary["files"][name] = {
        "filename": path.name,
        "columns": columns
    }


with open(
    DATA_DICTIONARY_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        data_dictionary,
        f,
        indent=2
    )


print()
print(
    "PASS: v1_data_dictionary.json created"
)


# ------------------------------------------------------------
# 5. SHA256 helper
# ------------------------------------------------------------

def sha256_file(path, chunk_size=1024 * 1024):

    digest = hashlib.sha256()

    with open(path, "rb") as f:

        while True:

            block = f.read(chunk_size)

            if not block:
                break

            digest.update(block)

    return digest.hexdigest()


# ------------------------------------------------------------
# 6. Create hash manifest from REAL files
# ------------------------------------------------------------

HASH_MANIFEST_PATH = (
    V1_DIR / "v1_artifact_hash_manifest.json"
)

files_to_hash = [
    *ARTIFACTS.values(),
    DATA_DICTIONARY_PATH,
]

hash_manifest = {
    "source": "Notebook 04 V1 generated artifacts",
    "algorithm": "SHA256",
    "files": {}
}

for path in files_to_hash:

    hash_manifest["files"][path.name] = {
        "sha256": sha256_file(path),
        "bytes": path.stat().st_size
    }


with open(
    HASH_MANIFEST_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        hash_manifest,
        f,
        indent=2
    )


print(
    "PASS: v1_artifact_hash_manifest.json created"
)


# ------------------------------------------------------------
# 7. Reload and verify both provenance artifacts
# ------------------------------------------------------------

with open(
    DATA_DICTIONARY_PATH,
    "r",
    encoding="utf-8"
) as f:

    reloaded_dictionary = json.load(f)


with open(
    HASH_MANIFEST_PATH,
    "r",
    encoding="utf-8"
) as f:

    reloaded_manifest = json.load(f)


if not reloaded_dictionary.get("files"):
    raise RuntimeError(
        "STOP — generated data dictionary is empty."
    )


if not reloaded_manifest.get("files"):
    raise RuntimeError(
        "STOP — generated hash manifest is empty."
    )


# ------------------------------------------------------------
# 8. Final status
# ------------------------------------------------------------

print()
print("=" * 80)
print("NOTEBOOK 05 — ARTIFACT DISCOVERY STATUS")
print("=" * 80)

print("Notebook 04 artifacts       : PASS")
print("Data dictionary             : PASS")
print("SHA256 hash manifest        : PASS")
print("Provenance reload check     : PASS")

print("=" * 80)
print("CELL 1: PASS")
print("=" * 80)

NOTEBOOK 05 — V1 ARTIFACT DISCOVERY
Project root: /content/drive/MyDrive/PINN_EV_Battery
V1 directory: /content/drive/MyDrive/PINN_EV_Battery/data/v1

Notebook 04 artifacts:
  PASS: cell_timeseries           2,569.27 MB
  PASS: particle_profiles         4.19 MB
  PASS: pack_timeseries           53.27 MB
  PASS: scenario_instances        0.01 MB
  PASS: s6_intervention_summary   0.00 MB
  PASS: generation_metadata       0.00 MB

PASS: v1_data_dictionary.json created
PASS: v1_artifact_hash_manifest.json created

NOTEBOOK 05 — ARTIFACT DISCOVERY STATUS
Notebook 04 artifacts       : PASS
Data dictionary             : PASS
SHA256 hash manifest        : PASS
Provenance reload check     : PASS
CELL 1: PASS


In [ ]:
# ============================================================
# CELL 3 — AUTHORITATIVE NOTEBOOK 04 SCHEMA RESOLUTION
# ============================================================

def resolve_column(
    columns,
    aliases,
    required=True,
    label="column"
):

    normalized = {
        re.sub(
            r"[^a-z0-9]+",
            "",
            str(column).lower()
        ): column
        for column in columns
    }

    for alias in aliases:

        key = re.sub(
            r"[^a-z0-9]+",
            "",
            str(alias).lower()
        )

        if key in normalized:
            return normalized[key]

    if required:
        raise RuntimeError(
            f"Missing required {label}. "
            f"Available columns: {list(columns)}"
        )

    return None


def header(path):

    return pd.read_csv(
        path,
        nrows=0
    )


cell_cols = header(
    ARTIFACTS["cell_timeseries"]
).columns.tolist()

scenario_cols = header(
    ARTIFACTS["scenario_instances"]
).columns.tolist()

profile_cols = header(
    ARTIFACTS["particle_profiles"]
).columns.tolist()

pack_cols = header(
    ARTIFACTS["pack_timeseries"]
).columns.tolist()


# ------------------------------------------------------------
# Actual Notebook 04 cell schema
# ------------------------------------------------------------

C = {

    "instance":
        resolve_column(
            cell_cols,
            [
                "scenario_instance_id",
                "instance_id"
            ],
            True,
            "scenario instance ID"
        ),

    "scenario":
        resolve_column(
            cell_cols,
            [
                "scenario_id"
            ],
            True,
            "scenario ID"
        ),

    "cell_id":
        resolve_column(
            cell_cols,
            [
                "cell_id"
            ],
            True,
            "cell ID"
        ),

    "time":
        resolve_column(
            cell_cols,
            [
                "time_s"
            ],
            True,
            "time"
        ),

    "split":
        resolve_column(
            cell_cols,
            [
                "data_split",
                "split"
            ],
            False
        ),

    "temperature":
        "cell_temperature_K",

    "surface_temperature":
        "surface_temperature_K",

    "voltage":
        "cell_voltage_V",

    "current":
        "cell_current_A",

    "soc":
        "soc_proxy",

    "soh":
        "capacity_retention_soh",

    "capacity":
        "capacity_Ah_reference_test",

    "resistance":
        "dc_pulse_resistance_proxy_ohm",

    "charge_rate":
        "requested_charge_rate_C",

    "ambient":
        "ambient_temperature_C",
}


# ------------------------------------------------------------
# Verify mapped physical columns
# ------------------------------------------------------------

for name, column in C.items():

    if column is None:
        continue

    if column not in cell_cols:

        raise RuntimeError(
            f"STOP — schema mapping for '{name}' "
            f"points to missing column '{column}'."
        )


print("Cell columns:")
print(cell_cols)

print()
print("Resolved authoritative schema:")
print(
    json.dumps(
        C,
        indent=2
    )
)

print()
print("Schema resolution: PASS")

Cell columns:
['scenario_id', 'scenario_instance_id', 'data_split', 'pack_topology', 'cell_id', 'state_kind', 'ageing_cycles', 'ambient_temperature_C', 'requested_charge_rate_C', 'conditioning_duration_s', 'time_s', 'cell_voltage_V', 'cell_current_A', 'cell_temperature_K', 'surface_temperature_K', 'ambient_temperature_K', 'soc_proxy', 'negative_particle_concentration_mean_mol_m3', 'positive_particle_concentration_mean_mol_m3', 'electrolyte_concentration_xavg_mol_m3', 'capacity_Ah_reference_test', 'capacity_retention_soh', 'dc_pulse_resistance_proxy_ohm', 'resistance_proxy_multiplier', 'pybamm_resistance_proxy_ohm', 'neg_sei_thickness', 'neg_crack_length', 'lam_negative']

Resolved authoritative schema:
{
  "instance": "scenario_instance_id",
  "scenario": "scenario_id",
  "cell_id": "cell_id",
  "time": "time_s",
  "split": "data_split",
  "temperature": "cell_temperature_K",
  "surface_temperature": "surface_temperature_K",
  "voltage": "cell_voltage_V",
  "current": "cell_current_A",

In [ ]:
# ============================================================
# CELL 4 — SCENARIO-INSTANCE INTEGRITY
# FINAL SCHEMA-SAFE VERSION
# ============================================================

print("=" * 80)
print("SCENARIO-INSTANCE INTEGRITY")
print("=" * 80)

scenarios = pd.read_csv(
    ARTIFACTS["scenario_instances"],
    low_memory=False
)

if scenarios.empty:
    raise RuntimeError(
        "STOP — scenario_instances.csv is empty."
    )

print("Scenario registry columns:")
print(list(scenarios.columns))

# ------------------------------------------------------------
# Authoritative Notebook 04 schema
# ------------------------------------------------------------

INSTANCE_COL = "scenario_instance_id"
SCENARIO_COL = "scenario_id"
SPLIT_COL = "split"
FAULT_COL = "fault_cells_json"
CELL_COUNT_COL = "cell_count"

required_columns = [
    INSTANCE_COL,
    SCENARIO_COL,
    SPLIT_COL,
    FAULT_COL,
    CELL_COUNT_COL,
]

missing = [
    c for c in required_columns
    if c not in scenarios.columns
]

if missing:
    raise RuntimeError(
        "STOP — scenario_instances.csv is missing required "
        f"columns: {missing}\n"
        f"Available columns: {list(scenarios.columns)}"
    )

# ------------------------------------------------------------
# Resolve number of cells FIRST
# ------------------------------------------------------------

cell_counts = pd.to_numeric(
    scenarios[CELL_COUNT_COL],
    errors="coerce"
)

if cell_counts.isna().any():
    raise RuntimeError(
        "STOP — cell_count contains NaN/non-numeric values."
    )

if not np.isfinite(
    cell_counts.to_numpy(dtype=float)
).all():
    raise RuntimeError(
        "STOP — cell_count contains non-finite values."
    )

unique_cell_counts = sorted(
    cell_counts.astype(int).unique().tolist()
)

if len(unique_cell_counts) != 1:
    raise RuntimeError(
        "STOP — inconsistent cell_count values: "
        f"{unique_cell_counts}"
    )

N_CELLS_VALIDATED = unique_cell_counts[0]

if N_CELLS_VALIDATED <= 0:
    raise RuntimeError(
        "STOP — invalid number of cells."
    )

# Important compatibility variable.
N_CELLS = N_CELLS_VALIDATED

print()
print(
    "Validated number of cells:",
    N_CELLS_VALIDATED
)

# ------------------------------------------------------------
# Instance uniqueness
# ------------------------------------------------------------

if scenarios[INSTANCE_COL].isna().any():
    raise RuntimeError(
        "STOP — scenario_instance_id contains missing values."
    )

if scenarios[INSTANCE_COL].duplicated().any():
    duplicates = (
        scenarios.loc[
            scenarios[INSTANCE_COL].duplicated(
                keep=False
            ),
            INSTANCE_COL
        ]
        .astype(str)
        .unique()
        .tolist()
    )

    raise RuntimeError(
        "STOP — duplicate scenario_instance_id values:\n"
        + "\n".join(
            f"  - {x}"
            for x in duplicates
        )
    )

# ------------------------------------------------------------
# Split validation
# ------------------------------------------------------------

scenarios[SPLIT_COL] = (
    scenarios[SPLIT_COL]
    .astype(str)
    .str.strip()
    .str.lower()
    .replace({"validation": "val"})
)

allowed_splits = {"train", "val", "test"}

invalid_splits = sorted(
    set(scenarios[SPLIT_COL]) - allowed_splits
)

if invalid_splits:
    raise RuntimeError(
        "STOP — invalid split values:\n"
        + "\n".join(
            f"  - {x}"
            for x in invalid_splits
        )
    )

# ------------------------------------------------------------
# Fault parser
# ------------------------------------------------------------

def parse_fault_cells(value):

    if pd.isna(value):
        return []

    if isinstance(
        value,
        (list, tuple, np.ndarray)
    ):
        raw_values = list(value)

    else:

        text = str(value).strip()

        if text in {
            "",
            "[]",
            "nan",
            "None",
            "null"
        }:
            return []

        try:
            parsed = json.loads(text)

            if isinstance(parsed, list):
                raw_values = parsed
            else:
                raw_values = [parsed]

        except Exception:

            raw_values = re.findall(
                r"\d+",
                text
            )

    faults = []

    for raw in raw_values:

        try:
            cell = int(raw)
        except Exception:
            continue

        if 1 <= cell <= N_CELLS_VALIDATED:
            faults.append(cell)

    return sorted(set(faults))


# ------------------------------------------------------------
# Build authoritative fault map
# ------------------------------------------------------------

fault_map = {}

for _, row in scenarios.iterrows():

    instance_id = str(
        row[INSTANCE_COL]
    )

    fault_map[instance_id] = parse_fault_cells(
        row[FAULT_COL]
    )

# ------------------------------------------------------------
# Validate fault locations
# ------------------------------------------------------------

invalid_fault_records = []

for instance_id, faults in fault_map.items():

    for cell_id in faults:

        if not (
            1 <= int(cell_id) <= N_CELLS_VALIDATED
        ):
            invalid_fault_records.append(
                (instance_id, cell_id)
            )

if invalid_fault_records:

    raise RuntimeError(
        "STOP — invalid fault-cell IDs:\n"
        + "\n".join(
            f"  {instance} -> cell {cell}"
            for instance, cell
            in invalid_fault_records
        )
    )

# ------------------------------------------------------------
# Summary
# ------------------------------------------------------------

print()
print("Scenario counts:")
print(
    scenarios[SCENARIO_COL]
    .value_counts()
    .sort_index()
)

print()
print("Provided split:")
print(
    scenarios[SPLIT_COL]
    .value_counts()
)

print()
print("Fault-location column:", FAULT_COL)

print(
    "Fault-location values are retained as "
    "evaluation ground truth only."
)

print(
    "Cell-37 benchmark records:",
    sum(
        37 in faults
        for faults in fault_map.values()
    )
)

print()
print("=" * 80)
print("SCENARIO-INSTANCE AUDIT: PASS")
print("=" * 80)

SCENARIO-INSTANCE INTEGRITY
Scenario registry columns:
['scenario_instance_id', 'scenario_id', 'split', 'pack_topology', 'cell_count', 'fault_cells_json', 'fault_count', 'fault_location_is_hidden_to_model', 'cell37_is_benchmark', 'healthy_reference_required', 'degradation_ground_truth_source', 'degradation_parameter_multipliers_used', 'threshold_status', 'localization_status', 'base_scenario_instance_id', 'candidate_charge_rate_C', 'winner_preselected']

Validated number of cells: 100

Scenario counts:
scenario_id
S1     1
S2     1
S3     1
S4     5
S5     5
S6    15
Name: count, dtype: int64

Provided split:
split
test     20
train     5
val       3
Name: count, dtype: int64

Fault-location column: fault_cells_json
Fault-location values are retained as evaluation ground truth only.
Cell-37 benchmark records: 7

SCENARIO-INSTANCE AUDIT: PASS


In [ ]:
# Large-file structural validation — CHUNKED

CHUNK = 100_000
usecols = list(dict.fromkeys([
    C["instance"], C["scenario"], C["cell_id"], C["time"]
] + [x for x in C.values() if x is not None]))

rows = 0
bad_core = 0
bad_cell = 0
bad_time = 0
instances = set()
scenarios_seen = set()

for chunk in pd.read_csv(
    ARTIFACTS["cell_timeseries"],
    usecols=usecols,
    chunksize=CHUNK,
    low_memory=False
):
    rows += len(chunk)
    bad_core += int(chunk[[C["instance"],C["scenario"],C["cell_id"],C["time"]]].isna().any(axis=1).sum())

    cid = pd.to_numeric(chunk[C["cell_id"]], errors="coerce")
    tm = pd.to_numeric(chunk[C["time"]], errors="coerce")

    bad_cell += int((cid.isna() | (cid < 1) | (cid > 100)).sum())
    bad_time += int((tm.isna() | (tm < 0)).sum())

    instances.update(chunk[C["instance"]].astype(str).unique())
    scenarios_seen.update(chunk[C["scenario"]].astype(str).unique())

if bad_core or bad_cell or bad_time:
    raise RuntimeError(
        f"Structural validation failed: missing_core={bad_core}, "
        f"bad_cell={bad_cell}, bad_time={bad_time}"
    )

print("Rows:", rows)
print("Instances:", len(instances))
print("Scenarios:", sorted(scenarios_seen))
print("Cell IDs: exactly within 1..100")
print("Large-file structural validation: PASS")


Rows: 7656014
Instances: 28
Scenarios: ['S1', 'S2', 'S3', 'S4', 'S5', 'S6']
Cell IDs: exactly within 1..100
Large-file structural validation: PASS


In [ ]:

# Exact 100-cell coverage per instance — CHUNKED

cells_by_instance = {}

for chunk in pd.read_csv(
    ARTIFACTS["cell_timeseries"],
    usecols=[C["instance"], C["cell_id"]],
    chunksize=CHUNK,
    low_memory=False
):
    for iid, g in chunk.groupby(C["instance"], sort=False):
        s = cells_by_instance.setdefault(str(iid), set())
        vals = pd.to_numeric(g[C["cell_id"]], errors="coerce").dropna().astype(int)
        s.update(vals.tolist())

expected = set(range(1,101))
bad = []

for iid, observed in cells_by_instance.items():
    missing = sorted(expected - observed)
    extra = sorted(observed - expected)
    if missing or extra:
        bad.append({"instance_id":iid,"missing":missing,"extra":extra})

if bad:
    raise RuntimeError("100-cell coverage failed:\n" + json.dumps(bad[:10], indent=2))

print("Instances checked:", len(cells_by_instance))
print("Every instance contains Cells 1–100.")
print("100-cell coverage: PASS")


Instances checked: 28
Every instance contains Cells 1–100.
100-cell coverage: PASS


In [ ]:

# Fault-location leakage audit

forbidden = ["faultcell","faultcells","faultlocation","defectcell","defectcells","criticalcell"]
possible = [c for c in cell_cols if any(x in re.sub(r"[^a-z0-9]","",c.lower()) for x in forbidden)]

ambiguous = [
    c for c in possible
    if not any(x in re.sub(r"[^a-z0-9]","",c.lower())
               for x in ["label","truth","ground","target"])
]

print("Potential fault-location columns:", possible)

if ambiguous:
    raise RuntimeError(
        "Potential fault-location leakage in model-facing cell_timeseries columns: "
        + str(ambiguous)
    )

print("Fault-location leakage gate: PASS")


Potential fault-location columns: []
Fault-location leakage gate: PASS


In [ ]:
# ============================================================
# HEALTHY / DEGRADED DESCRIPTIVE SEPARABILITY
# SCHEMA-AWARE FOR NOTEBOOK 04
# ============================================================

print("=" * 80)
print("HEALTHY / DEGRADED DESCRIPTIVE SEPARABILITY")
print("=" * 80)

# ------------------------------------------------------------
# Actual Notebook 04 schema
# ------------------------------------------------------------

SCENARIO_COL = "scenario_id"

FIELDS = {
    "temperature": "cell_temperature_K",
    "surface_temperature": "surface_temperature_K",
    "voltage": "cell_voltage_V",
    "current": "cell_current_A",
    "soc": "soc_proxy",
    "soh": "capacity_retention_soh",
    "capacity": "capacity_Ah_reference_test",
    "resistance": "dc_pulse_resistance_proxy_ohm",
}

# ------------------------------------------------------------
# Verify source columns
# ------------------------------------------------------------

available_fields = {
    name: column
    for name, column in FIELDS.items()
    if column in pd.read_csv(
        ARTIFACTS["cell_timeseries"],
        nrows=0
    ).columns
}

if not available_fields:
    raise RuntimeError(
        "STOP — no physical fields from the expected "
        "Notebook 04 schema were found."
    )

print(
    "Fields used:",
    available_fields
)

# ------------------------------------------------------------
# Chunked sampling
# ------------------------------------------------------------

usecols = [
    SCENARIO_COL
] + list(
    available_fields.values()
)

parts = []

for chunk in pd.read_csv(
    ARTIFACTS["cell_timeseries"],
    usecols=usecols,
    chunksize=CHUNK,
    low_memory=False
):

    if chunk.empty:
        continue

    sample_n = min(
        len(chunk),
        2000
    )

    parts.append(
        chunk.sample(
            n=sample_n,
            random_state=SEED
        )
    )

if not parts:
    raise RuntimeError(
        "STOP — no rows were sampled from cell_timeseries."
    )

sample = pd.concat(
    parts,
    ignore_index=True
)

# ------------------------------------------------------------
# Calculate descriptive statistics
# ------------------------------------------------------------

rows_out = []

for name, column in available_fields.items():

    values = pd.to_numeric(
        sample[column],
        errors="coerce"
    )

    tmp = pd.DataFrame({
        "scenario": sample[SCENARIO_COL].astype(str),
        "value": values
    }).dropna()

    if tmp.empty:
        continue

    for scenario_id, group in tmp.groupby(
        "scenario"
    ):

        rows_out.append({
            "field": name,
            "scenario": str(scenario_id),
            "n": int(len(group)),
            "mean": float(group["value"].mean()),
            "std": (
                float(group["value"].std())
                if len(group) > 1
                else 0.0
            ),
            "min": float(group["value"].min()),
            "max": float(group["value"].max()),
        })

separability = pd.DataFrame(
    rows_out
)

if separability.empty:
    raise RuntimeError(
        "STOP — separability summary contains no rows."
    )

display(separability)

separability.to_csv(
    V1_DIR / "05_separability_summary.csv",
    index=False
)

print()
print(
    "Separability audit: PASS"
)

print(
    "Rows in summary:",
    len(separability)
)

print("=" * 80)

HEALTHY / DEGRADED DESCRIPTIVE SEPARABILITY
Fields used: {'temperature': 'cell_temperature_K', 'surface_temperature': 'surface_temperature_K', 'voltage': 'cell_voltage_V', 'current': 'cell_current_A', 'soc': 'soc_proxy', 'soh': 'capacity_retention_soh', 'capacity': 'capacity_Ah_reference_test', 'resistance': 'dc_pulse_resistance_proxy_ohm'}


,field,scenario,n,mean,std,min,max
0,temperature,S1,5379,310.656570,4.348047e+00,298.150000,314.509087
1,temperature,S2,5457,318.350270,1.222201e+01,298.150000,339.783267
2,temperature,S3,5389,315.775697,8.658074e+00,298.150000,328.586080
3,temperature,S4,26970,310.686206,4.374128e+00,298.150000,315.000230
4,temperature,S5,27572,319.161604,1.299009e+01,298.150000,341.128110
5,temperature,S6,83233,316.121348,9.818517e+00,298.150000,341.128110
6,surface_temperature,S1,5379,302.354709,1.615894e+00,298.150000,303.792078
7,surface_temperature,S2,5457,304.468975,3.914468e+00,298.150000,311.888981
8,surface_temperature,S3,5389,310.161566,9.019552e+00,298.150000,321.678650
9,surface_temperature,S4,26970,302.367267,1.624237e+00,298.150000,303.806718



Separability audit: PASS
Rows in summary: 48


In [ ]:
# ============================================================
# SPLIT INTEGRITY CHECK
# REPLACEMENT CELL
# ============================================================

print("=" * 80)
print("SPLIT INTEGRITY CHECK")
print("=" * 80)

# ------------------------------------------------------------
# 1. Load scenario registry
# ------------------------------------------------------------

scenarios = pd.read_csv(
    ARTIFACTS["scenario_instances"]
)

if scenarios.empty:
    raise RuntimeError(
        "STOP — scenario_instances.csv is empty."
    )

# ------------------------------------------------------------
# 2. Resolve ACTUAL Notebook 04 column names
# ------------------------------------------------------------

if "scenario_instance_id" in scenarios.columns:
    INSTANCE_COL = "scenario_instance_id"
elif "instance_id" in scenarios.columns:
    INSTANCE_COL = "instance_id"
else:
    raise RuntimeError(
        "STOP — scenario registry has neither "
        "'scenario_instance_id' nor 'instance_id'."
    )

if "scenario_id" not in scenarios.columns:
    raise RuntimeError(
        "STOP — scenario registry has no scenario_id column."
    )

# Notebook 04 actually uses "split"
# according to the registry output you posted.
if "split" in scenarios.columns:
    SPLIT_COL = "split"
elif "data_split" in scenarios.columns:
    SPLIT_COL = "data_split"
else:
    raise RuntimeError(
        "STOP — scenario registry has neither "
        "'split' nor 'data_split'."
    )

print("Resolved scenario schema:")
print(f"  instance : {INSTANCE_COL}")
print(f"  scenario : scenario_id")
print(f"  split    : {SPLIT_COL}")

# ------------------------------------------------------------
# 3. Create normalized validation dataframe
# ------------------------------------------------------------

splits = scenarios[
    [
        INSTANCE_COL,
        "scenario_id",
        SPLIT_COL
    ]
].copy()

splits.columns = [
    "instance_id",
    "scenario_id",
    "split"
]

# ------------------------------------------------------------
# 4. Basic integrity checks
# ------------------------------------------------------------

if splits["instance_id"].isna().any():
    raise RuntimeError(
        "STOP — scenario registry contains "
        "missing instance IDs."
    )

if splits["instance_id"].duplicated().any():
    duplicate_ids = (
        splits.loc[
            splits["instance_id"].duplicated(keep=False),
            "instance_id"
        ]
        .astype(str)
        .unique()
        .tolist()
    )

    raise RuntimeError(
        "STOP — duplicate scenario instance IDs found:\n"
        + "\n".join(
            f"  - {x}"
            for x in duplicate_ids
        )
    )

# ------------------------------------------------------------
# 5. Validate split values
# ------------------------------------------------------------

splits["split"] = (
    splits["split"]
    .astype(str)
    .str.strip()
    .str.lower()
)

allowed_splits = {
    "train",
    "val",
    "validation",
    "test"
}

invalid_splits = sorted(
    set(splits["split"]) - allowed_splits
)

if invalid_splits:
    raise RuntimeError(
        "STOP — invalid split labels found:\n"
        + "\n".join(
            f"  - {x}"
            for x in invalid_splits
        )
    )

# Normalize validation -> val
splits["split"] = splits["split"].replace(
    {
        "validation": "val"
    }
)

# ------------------------------------------------------------
# 6. Print split distribution
# ------------------------------------------------------------

print()
print("Split distribution:")
print(
    splits["split"]
    .value_counts()
    .sort_index()
)

print()
print("Scenario × split distribution:")

scenario_split_table = pd.crosstab(
    splits["scenario_id"],
    splits["split"]
)

display(
    scenario_split_table
)

# ------------------------------------------------------------
# 7. Check that every instance belongs to exactly one split
# ------------------------------------------------------------

split_counts = (
    splits
    .groupby("instance_id")["split"]
    .nunique()
)

if (split_counts > 1).any():
    bad_ids = (
        split_counts[
            split_counts > 1
        ]
        .index
        .astype(str)
        .tolist()
    )

    raise RuntimeError(
        "STOP — scenario instances occur in "
        "multiple dataset splits:\n"
        + "\n".join(
            f"  - {x}"
            for x in bad_ids
        )
    )

# ------------------------------------------------------------
# 8. Check expected split coverage
# ------------------------------------------------------------

required_split_names = {
    "train",
    "val",
    "test"
}

present_splits = set(
    splits["split"].unique()
)

missing_splits = (
    required_split_names - present_splits
)

if missing_splits:
    raise RuntimeError(
        "STOP — required dataset split(s) missing:\n"
        + "\n".join(
            f"  - {x}"
            for x in sorted(missing_splits)
        )
    )

# ------------------------------------------------------------
# 9. Save normalized split registry
# ------------------------------------------------------------

split_output = (
    V1_DIR /
    "05_split_integrity_summary.csv"
)

splits.to_csv(
    split_output,
    index=False
)

# ------------------------------------------------------------
# 10. Final status
# ------------------------------------------------------------

print()
print(
    f"Total scenario instances: "
    f"{len(splits):,}"
)

print(
    f"Unique scenario instances: "
    f"{splits['instance_id'].nunique():,}"
)

print(
    f"Unique scenarios: "
    f"{splits['scenario_id'].nunique():,}"
)

print()
print(
    f"Split integrity output: {split_output}"
)

print()
print("=" * 80)
print("SPLIT INTEGRITY AUDIT: PASS")
print("=" * 80)

SPLIT INTEGRITY CHECK
Resolved scenario schema:
  instance : scenario_instance_id
  scenario : scenario_id
  split    : split

Split distribution:
split
test     20
train     5
val       3
Name: count, dtype: int64

Scenario × split distribution:


split,test,train,val
scenario_id,,,
S1,0,1,0
S2,0,1,0
S3,0,0,1
S4,2,2,1
S5,3,1,1
S6,15,0,0



Total scenario instances: 28
Unique scenario instances: 28
Unique scenarios: 6

Split integrity output: /content/drive/MyDrive/PINN_EV_Battery/data/v1/05_split_integrity_summary.csv

SPLIT INTEGRITY AUDIT: PASS


In [ ]:
# ============================================================
# CELL 10 — TRAJECTORY → SPLIT INTEGRITY
# FINAL SCHEMA-SAFE VERSION
# ============================================================

print("=" * 80)
print("TRAJECTORY → SPLIT INTEGRITY")
print("=" * 80)

# ------------------------------------------------------------
# Authoritative split mapping
# ------------------------------------------------------------

if "splits" not in globals():

    if "scenarios" not in globals():
        scenarios = pd.read_csv(
            ARTIFACTS["scenario_instances"],
            low_memory=False
        )

    splits = scenarios[
        [
            "scenario_instance_id",
            "scenario_id",
            "split"
        ]
    ].copy()

    splits.columns = [
        "instance_id",
        "scenario_id",
        "split"
    ]

# Normalize IDs
splits["instance_id"] = (
    splits["instance_id"]
    .astype(str)
    .str.strip()
)

splits["split"] = (
    splits["split"]
    .astype(str)
    .str.strip()
    .str.lower()
    .replace({"validation": "val"})
)

split_map = dict(
    zip(
        splits["instance_id"],
        splits["split"]
    )
)

if not split_map:
    raise RuntimeError(
        "STOP — split map is empty."
    )

# ------------------------------------------------------------
# Check every cell-timeseries trajectory
# ------------------------------------------------------------

unknown = set()

trajectory_instances = set()

for chunk in pd.read_csv(
    ARTIFACTS["cell_timeseries"],
    usecols=["scenario_instance_id"],
    chunksize=CHUNK,
    low_memory=False
):

    ids = (
        chunk["scenario_instance_id"]
        .astype(str)
        .str.strip()
    )

    trajectory_instances.update(
        ids.unique().tolist()
    )

    missing = ids[
        ~ids.isin(split_map)
    ]

    if not missing.empty:
        unknown.update(
            missing.unique().tolist()
        )

if unknown:
    raise RuntimeError(
        "STOP — cell_timeseries contains scenario "
        "instance IDs absent from the split registry:\n"
        + "\n".join(
            f"  - {x}"
            for x in sorted(unknown)[:50]
        )
    )

# ------------------------------------------------------------
# Check coverage
# ------------------------------------------------------------

registry_ids = set(
    split_map.keys()
)

missing_trajectories = (
    registry_ids - trajectory_instances
)

if missing_trajectories:
    raise RuntimeError(
        "STOP — scenario registry contains instances "
        "with no cell_timeseries trajectory:\n"
        + "\n".join(
            f"  - {x}"
            for x in sorted(missing_trajectories)[:50]
        )
    )

print(
    "Scenario instances in registry:",
    len(registry_ids)
)

print(
    "Scenario instances in cell_timeseries:",
    len(trajectory_instances)
)

print()
print(
    "Every trajectory maps to exactly one "
    "known dataset split."
)

print()
print("=" * 80)
print("TRAJECTORY → SPLIT INTEGRITY: PASS")
print("=" * 80)

TRAJECTORY → SPLIT INTEGRITY
Scenario instances in registry: 28
Scenario instances in cell_timeseries: 28

Every trajectory maps to exactly one known dataset split.

TRAJECTORY → SPLIT INTEGRITY: PASS


In [ ]:

# Future-target specification — no future leakage

future_spec = {
    "prediction_horizons_seconds":[60,300,600],
    "group_keys":["instance_id","cell_id"],
    "sort_key":"time_s",
    "target_rule":(
        "For timestamp t and horizon H, target is the first observation "
        "at time >= t+H within the same instance_id and cell_id."
    ),
    "drop_rule":"Rows without an available future observation are excluded for that horizon.",
    "split_rule":"Train/validation/test are separated at scenario-instance level.",
    "fault_location_rule":"Fault location is evaluation ground truth only, never an input feature.",
    "status":"SPECIFICATION_READY_FOR_NOTEBOOK_06"
}

future_path = V1_DIR/"05_future_target_specification.json"
future_path.write_text(json.dumps(future_spec,indent=2))
print(json.dumps(future_spec,indent=2))


{
  "prediction_horizons_seconds": [
    60,
    300,
    600
  ],
  "group_keys": [
    "instance_id",
    "cell_id"
  ],
  "sort_key": "time_s",
  "target_rule": "For timestamp t and horizon H, target is the first observation at time >= t+H within the same instance_id and cell_id.",
  "drop_rule": "Rows without an available future observation are excluded for that horizon.",
  "split_rule": "Train/validation/test are separated at scenario-instance level.",
  "fault_location_rule": "Fault location is evaluation ground truth only, never an input feature.",
  "status": "SPECIFICATION_READY_FOR_NOTEBOOK_06"
}


In [ ]:
# ============================================================
# CELL 12 — PARTICLE PROFILE VALIDATION
# FINAL NOTEBOOK 04 SCHEMA
# ============================================================

print("=" * 80)
print("PARTICLE PROFILE VALIDATION")
print("=" * 80)

profiles = pd.read_csv(
    ARTIFACTS["particle_profiles"],
    low_memory=False
)

if profiles.empty:
    raise RuntimeError(
        "STOP — particle_profiles.csv is empty."
    )

print("Particle profile columns:")
print(list(profiles.columns))

# ------------------------------------------------------------
# Authoritative Notebook 04 schema
# ------------------------------------------------------------

R_NORM_COL = "radial_position_normalized"
R_M_COL = "radial_position_m"
TIME_COL = "time_s"

CONC_COL = "negative_particle_concentration"

required_columns = [
    R_NORM_COL,
    R_M_COL,
    TIME_COL,
    CONC_COL,
]

missing = [
    c
    for c in required_columns
    if c not in profiles.columns
]

if missing:
    raise RuntimeError(
        "STOP — particle_profiles.csv is missing "
        f"required columns: {missing}\n\n"
        "Available columns:\n"
        + "\n".join(
            f"  - {c}"
            for c in profiles.columns
        )
    )

# ------------------------------------------------------------
# Convert numeric columns
# ------------------------------------------------------------

r_norm = pd.to_numeric(
    profiles[R_NORM_COL],
    errors="coerce"
)

r_m = pd.to_numeric(
    profiles[R_M_COL],
    errors="coerce"
)

time_s = pd.to_numeric(
    profiles[TIME_COL],
    errors="coerce"
)

concentration = pd.to_numeric(
    profiles[CONC_COL],
    errors="coerce"
)

# ------------------------------------------------------------
# NaN checks
# ------------------------------------------------------------

for name, series in {
    "normalized radius": r_norm,
    "physical radius": r_m,
    "time": time_s,
    "concentration": concentration,
}.items():

    if series.isna().any():

        raise RuntimeError(
            f"STOP — particle profile {name} "
            "contains NaN/non-numeric values."
        )

# ------------------------------------------------------------
# Finite checks
# ------------------------------------------------------------

for name, series in {
    "normalized radius": r_norm,
    "physical radius": r_m,
    "time": time_s,
    "concentration": concentration,
}.items():

    if not np.isfinite(
        series.to_numpy(dtype=float)
    ).all():

        raise RuntimeError(
            f"STOP — particle profile {name} "
            "contains non-finite values."
        )

# ------------------------------------------------------------
# Range checks
# ------------------------------------------------------------

if (
    (r_norm < 0).any()
    or
    (r_norm > 1).any()
):

    raise RuntimeError(
        "STOP — normalized radius must be in [0, 1]."
    )

if (r_m < 0).any():

    raise RuntimeError(
        "STOP — physical radius contains negative values."
    )

if (time_s < 0).any():

    raise RuntimeError(
        "STOP — particle profile contains negative time."
    )

if (concentration < 0).any():

    raise RuntimeError(
        "STOP — particle concentration contains "
        "negative values."
    )

# ------------------------------------------------------------
# Radial structure validation
# ------------------------------------------------------------

unique_radial_points = int(
    r_norm.nunique()
)

if unique_radial_points < 3:

    raise RuntimeError(
        "STOP — fewer than 3 distinct radial positions."
    )

# ------------------------------------------------------------
# Expected normalized radius coverage
# ------------------------------------------------------------

rmin = float(r_norm.min())
rmax = float(r_norm.max())

if not np.isclose(
    rmin,
    0.0,
    atol=1e-8
):

    raise RuntimeError(
        f"STOP — normalized radius minimum is {rmin}, "
        "expected approximately 0."
    )

if not np.isclose(
    rmax,
    1.0,
    atol=1e-8
):

    raise RuntimeError(
        f"STOP — normalized radius maximum is {rmax}, "
        "expected approximately 1."
    )

# ------------------------------------------------------------
# Report
# ------------------------------------------------------------

print()
print(
    "Particle profile rows:",
    len(profiles)
)

print(
    "Distinct radial points:",
    unique_radial_points
)

print(
    "Normalized radius:",
    rmin,
    "to",
    rmax
)

print(
    "Physical radius [m]:",
    float(r_m.min()),
    "to",
    float(r_m.max())
)

print(
    "Time [s]:",
    float(time_s.min()),
    "to",
    float(time_s.max())
)

print(
    "Concentration [mol/m3]:",
    float(concentration.min()),
    "to",
    float(concentration.max())
)

print()
print("=" * 80)
print("PARTICLE-PROFILE VALIDATION: PASS")
print("=" * 80)

PARTICLE PROFILE VALIDATION
Particle profile columns:
['scenario_id', 'scenario_instance_id', 'data_split', 'pack_topology', 'cell_id', 'state_kind', 'time_s', 'time_normalized', 'radial_position_normalized', 'radial_position_m', 'negative_particle_concentration', 'negative_radial_stress', 'negative_tangential_stress']

Particle profile rows: 26901
Distinct radial points: 21
Normalized radius: 0.0 to 1.0
Physical radius [m]: 0.0 to 5.86e-06
Time [s]: 0.0 to 178170.49763841563
Concentration [mol/m3]: 1208.1462224658358 to 30053.911836777934

PARTICLE-PROFILE VALIDATION: PASS


In [ ]:

# Pack artifact sanity check

pack = pd.read_csv(ARTIFACTS["pack_timeseries"], nrows=200_000)
if pack.empty:
    raise RuntimeError("pack_timeseries.csv is empty.")

print("Pack sample rows:",len(pack))
print("Pack columns:",list(pack.columns))
print("Pack artifact: PASS")


Pack sample rows: 200000
Pack columns: ['scenario_id', 'scenario_instance_id', 'data_split', 'pack_topology', 'cell_id', 'time_s', 'pack_voltage_V', 'pack_current_A', 'cell_voltage_V_interp']
Pack artifact: PASS


In [ ]:
# ============================================================
# CELL-37 AND NON-37 LOCALIZATION COVERAGE
# ============================================================

print("=" * 80)
print("CELL-37 AND NON-37 LOCALIZATION COVERAGE")
print("=" * 80)

# ------------------------------------------------------------
# Use the schema already validated above
# ------------------------------------------------------------

INSTANCE_COL = "scenario_instance_id"
SCENARIO_COL = "scenario_id"
FAULT_COL = "fault_cells_json"

required = [
    INSTANCE_COL,
    SCENARIO_COL,
    FAULT_COL,
]

missing = [
    c for c in required
    if c not in scenarios.columns
]

if missing:
    raise RuntimeError(
        "STOP — localization audit missing columns: "
        f"{missing}"
    )

# ------------------------------------------------------------
# Build fault map
# ------------------------------------------------------------

fault_map = {}

for _, row in scenarios.iterrows():

    instance_id = str(
        row[INSTANCE_COL]
    )

    faults = parse_fault_cells(
        row[FAULT_COL]
    )

    fault_map[instance_id] = faults


# ------------------------------------------------------------
# Coverage
# ------------------------------------------------------------

cell37 = [
    instance_id
    for instance_id, faults in fault_map.items()
    if 37 in faults
]

non37 = [
    instance_id
    for instance_id, faults in fault_map.items()
    if faults and any(
        int(cell) != 37
        for cell in faults
    )
]

print()
print(
    "Instance column :",
    INSTANCE_COL
)

print(
    "Scenario column :",
    SCENARIO_COL
)

print(
    "Fault column    :",
    FAULT_COL
)

print()
print(
    "Cell-37 instances:",
    cell37
)

print(
    "Non-37 fault instances:",
    non37
)

# ------------------------------------------------------------
# Required coverage checks
# ------------------------------------------------------------

if len(cell37) == 0:
    raise RuntimeError(
        "STOP — Cell-37 benchmark is missing."
    )

if len(non37) == 0:
    raise RuntimeError(
        "STOP — non-37 fault cases are missing."
    )

# ------------------------------------------------------------
# Final
# ------------------------------------------------------------

print()
print(
    "Cell-37 benchmark coverage: PASS"
)

print(
    "Non-37 fault-location coverage: PASS"
)

print(
    "Localization ground-truth coverage: PASS"
)

print("=" * 80)

CELL-37 AND NON-37 LOCALIZATION COVERAGE

Instance column : scenario_instance_id
Scenario column : scenario_id
Fault column    : fault_cells_json

Cell-37 instances: ['S4_CELL037_BENCH', 'S5_CELL037_BENCH', 'S6_S5_CELL037_BENCH_2p0C', 'S6_S5_CELL037_BENCH_1p5C', 'S6_S5_CELL037_BENCH_1p0C', 'S6_S5_CELL037_BENCH_0p8C', 'S6_S5_CELL037_BENCH_0p4C']
Non-37 fault instances: ['S4_CELL004', 'S4_CELL019', 'S5_CELL004', 'S4_CELL055', 'S5_CELL073', 'S4_CELL082', 'S5_CELL091', 'S5_CELL004_082', 'S6_S5_CELL091_2p0C', 'S6_S5_CELL091_1p5C', 'S6_S5_CELL091_1p0C', 'S6_S5_CELL091_0p8C', 'S6_S5_CELL091_0p4C', 'S6_S5_CELL004_082_2p0C', 'S6_S5_CELL004_082_1p5C', 'S6_S5_CELL004_082_1p0C', 'S6_S5_CELL004_082_0p8C', 'S6_S5_CELL004_082_0p4C']

Cell-37 benchmark coverage: PASS
Non-37 fault-location coverage: PASS
Localization ground-truth coverage: PASS


In [ ]:

# External experimental-data compatibility contract

external_contract = {
    "status":"READY_FOR_FUTURE_EXTERNAL_VALIDATION",
    "required_fields":[
        {"field":"time_s","unit":"s"},
        {"field":"cell_id","unit":"integer"},
        {"field":"voltage_v","unit":"V"},
        {"field":"current_a","unit":"A"},
        {"field":"temperature_c","unit":"degC"}
    ],
    "recommended_health_fields":[
        {"field":"soc","unit":"fraction"},
        {"field":"capacity_ah","unit":"Ah"},
        {"field":"resistance_ohm","unit":"ohm"},
        {"field":"cycle_number","unit":"cycle"}
    ],
    "metadata_required":[
        "chemistry","nominal_capacity","nominal_voltage",
        "sampling_interval","ambient_condition",
        "charge_discharge_protocol","sensor_accuracy",
        "end_of_life_definition"
    ],
    "rule":"Map by physical meaning and unit; never silently substitute quantities."
}

(V1_DIR/"05_external_data_compatibility.json").write_text(
    json.dumps(external_contract,indent=2)
)
print(json.dumps(external_contract,indent=2))


{
  "status": "READY_FOR_FUTURE_EXTERNAL_VALIDATION",
  "required_fields": [
    {
      "field": "time_s",
      "unit": "s"
    },
    {
      "field": "cell_id",
      "unit": "integer"
    },
    {
      "field": "voltage_v",
      "unit": "V"
    },
    {
      "field": "current_a",
      "unit": "A"
    },
    {
      "field": "temperature_c",
      "unit": "degC"
    }
  ],
  "recommended_health_fields": [
    {
      "field": "soc",
      "unit": "fraction"
    },
    {
      "field": "capacity_ah",
      "unit": "Ah"
    },
    {
      "field": "resistance_ohm",
      "unit": "ohm"
    },
    {
      "field": "cycle_number",
      "unit": "cycle"
    }
  ],
  "metadata_required": [
    "chemistry",
    "nominal_capacity",
    "nominal_voltage",
    "sampling_interval",
    "ambient_condition",
    "charge_discharge_protocol",
    "sensor_accuracy",
    "end_of_life_definition"
  ],
  "rule": "Map by physical meaning and unit; never silently substitute quantities."
}


In [ ]:
# ============================================================
# CELL 16 — FINAL NOTEBOOK 05 REPORT
# ============================================================

print("=" * 80)
print("NOTEBOOK 05 — FINAL VALIDATION REPORT")
print("=" * 80)

# ------------------------------------------------------------
# Resolve final counts safely
# ------------------------------------------------------------

cell_rows = int(
    rows
) if "rows" in globals() else None

particle_rows = int(
    len(profiles)
) if "profiles" in globals() else None

scenario_rows = int(
    len(scenarios)
) if "scenarios" in globals() else None

# ------------------------------------------------------------
# Resolve split output path
# ------------------------------------------------------------

split_path = (
    V1_DIR /
    "05_split_integrity_summary.csv"
)

# ------------------------------------------------------------
# Verify required outputs
# ------------------------------------------------------------

required_outputs = {
    "split_manifest": split_path,
    "separability_summary":
        V1_DIR / "05_separability_summary.csv",
    "future_target_specification":
        V1_DIR / "05_future_target_specification.json",
    "external_data_contract":
        V1_DIR / "05_external_data_compatibility.json",
}

missing_outputs = [
    name
    for name, path in required_outputs.items()
    if not path.exists()
]

if missing_outputs:
    raise RuntimeError(
        "STOP — Notebook 05 required outputs are missing:\n"
        + "\n".join(
            f"  - {x}"
            for x in missing_outputs
        )
    )

# ------------------------------------------------------------
# Final report
# ------------------------------------------------------------

report = {
    "notebook":
        "05_V1_DatasetValidation_GroundTruth.ipynb",

    "status":
        "PASS",

    "gpu_required":
        False,

    "large_file_strategy":
        "chunked",

    "checks": {
        "artifacts_exist": True,
        "notebook04_generation_pass": True,
        "core_schema_pass": True,
        "100_cells_per_instance": True,
        "fault_location_leakage_gate_pass": True,
        "instance_level_split_created": True,
        "split_integrity_pass": True,
        "particle_profiles_pass": True,
        "cell37_present": True,
        "non37_faults_present": True,
        "future_target_specification_ready": True,
        "external_data_contract_ready": True,
        "operational_safety_thresholds_validated": False
    },

    "row_count_cell_timeseries":
        cell_rows,

    "row_count_particle_profiles":
        particle_rows,

    "scenario_instances":
        scenario_rows,

    "next_notebook":
        "06_V1_PINN_Training_ModelValidation.ipynb"
}

report_path = (
    V1_DIR /
    "05_dataset_validation_report.json"
)

report_path.write_text(
    json.dumps(
        report,
        indent=2
    ),
    encoding="utf-8"
)

summary = pd.DataFrame(
    [
        {
            "check": key,
            "status": value
        }
        for key, value
        in report["checks"].items()
    ]
)

summary_path = (
    V1_DIR /
    "05_validation_summary.csv"
)

summary.to_csv(
    summary_path,
    index=False
)

# ------------------------------------------------------------
# Final status
# ------------------------------------------------------------

print()
print("=" * 80)
print("NOTEBOOK 05 — FINAL STATUS")
print("=" * 80)

print("STATUS: PASS")

print(
    "cell_timeseries rows:",
    cell_rows
)

print(
    "particle_profiles rows:",
    particle_rows
)

print(
    "scenario instances:",
    scenario_rows
)

print(
    "split manifest:",
    split_path
)

print(
    "validation report:",
    report_path
)

print(
    "validation summary:",
    summary_path
)

print(
    "operational safety thresholds validated: FALSE"
)

print()
print(
    "NEXT: 06_V1_PINN_Training_ModelValidation.ipynb"
)

print("=" * 80)

NOTEBOOK 05 — FINAL VALIDATION REPORT

NOTEBOOK 05 — FINAL STATUS
STATUS: PASS
cell_timeseries rows: 7656014
particle_profiles rows: 26901
scenario instances: 28
split manifest: /content/drive/MyDrive/PINN_EV_Battery/data/v1/05_split_integrity_summary.csv
validation report: /content/drive/MyDrive/PINN_EV_Battery/data/v1/05_dataset_validation_report.json
validation summary: /content/drive/MyDrive/PINN_EV_Battery/data/v1/05_validation_summary.csv
operational safety thresholds validated: FALSE

NEXT: 06_V1_PINN_Training_ModelValidation.ipynb


In [3]:
from pathlib import Path
import json
import sys
import os

ROOT = Path("/content/drive/MyDrive/PINN_EV_Battery")
V1 = ROOT / "data" / "v1"

print("=" * 60)
print("NOTEBOOK 05 — PRE-FLIGHT DIAGNOSTIC")
print("=" * 60)

# 1. Runtime
print("\n[1] RUNTIME")
print("Python:", sys.version.split()[0])

try:
    import psutil
    print("Available RAM (GB):",
          round(psutil.virtual_memory().available / 1024**3, 2))
except ImportError:
    print("RAM information unavailable")

try:
    import torch
    print("PyTorch installed:", True)
    print("CUDA available:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("GPU:", torch.cuda.get_device_name(0))
except ImportError:
    print("PyTorch installed:", False)

# 2. Drive and required artifacts
print("\n[2] DRIVE AND ARTIFACTS")
print("Project root exists:", ROOT.exists())
print("V1 directory exists:", V1.exists())

required_files = [
    "cell_timeseries.csv",
    "canonical/cell_timeseries_canonical.csv",
    "pack_timeseries.csv",
    "particle_profiles.csv",
    "scenario_instances.csv",
    "s6_intervention_summary.csv",
]

for name in required_files:
    p = V1 / name
    print(
        f"{name}:",
        "EXISTS" if p.is_file() else "MISSING",
        f"({p.stat().st_size / 1e6:.2f} MB)" if p.is_file() else ""
    )

# 3. Saved validation reports
print("\n[3] EXISTING VALIDATION REPORTS")

report_names = [
    "v1_generation_validation.json",
    "final_computed_validation.json",
    "05_dataset_validation_report.json",
    "05_split_integrity_summary.csv",
    "05_separability_summary.csv",
    "05_particle_profile_validation.csv",
    "05_physical_distribution_audit.csv",
    "05_event_boundary_audit.json",
    "05_duplicate_diagnostic.json",
    "05_future_target_specification.json",
]

for name in report_names:
    p = V1 / name
    print(f"{name}: {'EXISTS' if p.is_file() else 'MISSING'}")
    if p.is_file() and p.suffix == ".json":
        try:
            data = json.loads(p.read_text())
            print("  status:", data.get("status", "not specified"))
            print("  notebook:", data.get("notebook", "not specified"))
            if "checks" in data:
                failed = [
                    k for k, v in data["checks"].items()
                    if v is False
                ]
                print("  failed checks:", failed or "none explicitly false")
        except Exception as e:
            print("  JSON read error:", str(e))

# 4. Existing variables
print("\n[4] NOTEBOOK VARIABLES")
for name in [
    "cell_timeseries", "particle_profiles", "pack_timeseries",
    "SCENARIO_INSTANCES", "validation_report"
]:
    print(f"{name}: {'AVAILABLE' if name in globals() else 'NOT IN MEMORY'}")

print("\nDiagnostic complete. No simulations or dataset writes were performed.")

NOTEBOOK 05 — PRE-FLIGHT DIAGNOSTIC

[1] RUNTIME
Python: 3.13.18
Available RAM (GB): 11.24
PyTorch installed: True
CUDA available: False

[2] DRIVE AND ARTIFACTS
Project root exists: True
V1 directory exists: True
cell_timeseries.csv: EXISTS (2694.07 MB)
canonical/cell_timeseries_canonical.csv: EXISTS (1238.71 MB)
pack_timeseries.csv: EXISTS (55.86 MB)
particle_profiles.csv: EXISTS (4.40 MB)
scenario_instances.csv: EXISTS (0.01 MB)
s6_intervention_summary.csv: EXISTS (0.00 MB)

[3] EXISTING VALIDATION REPORTS
v1_generation_validation.json: EXISTS
  status: PASS
  notebook: not specified
  failed checks: none explicitly false
final_computed_validation.json: EXISTS
  status: PASS
  notebook: not specified
05_dataset_validation_report.json: EXISTS
  status: PASS
  notebook: 05_V1_DatasetValidation_GroundTruth.ipynb
  failed checks: ['operational_safety_thresholds_validated']
05_split_integrity_summary.csv: EXISTS
05_separability_summary.csv: EXISTS
05_particle_profile_validation.csv: EXIS

In [4]:
from pathlib import Path
import json
import pandas as pd

V1 = Path("/content/drive/MyDrive/PINN_EV_Battery/data/v1")

reports = [
    "05_validation_summary.csv",
    "05_split_integrity_summary.csv",
    "05_separability_summary.csv",
    "05_particle_profile_validation.csv",
    "05_physical_distribution_audit.csv",
    "06_future_target_timing_audit.csv",
]

json_reports = [
    "05_duplicate_diagnostic.json",
    "05_event_boundary_audit.json",
    "05_external_data_compatibility.json",
    "05_future_target_specification.json",
    "05_V1_model_input_contract.json",
]

print("=" * 65)
print("NOTEBOOK 05 — EXISTING VALIDATION RESULTS")
print("=" * 65)

for name in reports:
    path = V1 / name
    print(f"\n--- {name} ---")
    if path.exists():
        try:
            df = pd.read_csv(path)
            print("Shape:", df.shape)
            print(df.to_string(index=False, max_rows=40))
        except Exception as e:
            print("READ ERROR:", repr(e))
    else:
        print("MISSING")

for name in json_reports:
    path = V1 / name
    print(f"\n--- {name} ---")
    if path.exists():
        try:
            print(json.dumps(json.loads(path.read_text()), indent=2))
        except Exception as e:
            print("READ ERROR:", repr(e))
    else:
        print("MISSING")

print("\nInspection complete. No large dataset was loaded or modified.")

NOTEBOOK 05 — EXISTING VALIDATION RESULTS

--- 05_validation_summary.csv ---
Shape: (13, 2)
                                  check  status
                        artifacts_exist    True
             notebook04_generation_pass    True
                       core_schema_pass    True
                 100_cells_per_instance    True
       fault_location_leakage_gate_pass    True
           instance_level_split_created    True
                   split_integrity_pass    True
                 particle_profiles_pass    True
                         cell37_present    True
                   non37_faults_present    True
      future_target_specification_ready    True
           external_data_contract_ready    True
operational_safety_thresholds_validated   False

--- 05_split_integrity_summary.csv ---
Shape: (28, 3)
    scenario_instance_id scenario_id split
              S1_HEALTHY          S1 train
              S2_HEALTHY          S2 train
              S3_HEALTHY          S3   val
         

In [5]:
from pathlib import Path
import pandas as pd
import numpy as np

V1 = Path("/content/drive/MyDrive/PINN_EV_Battery/data/v1")

FILES = {
    "raw": V1 / "cell_timeseries.csv",
    "canonical": V1 / "canonical" / "cell_timeseries_canonical.csv",
}

CHUNK_SIZE = 200_000

def audit_csv(label, path):
    print(f"\n{'=' * 65}\n{label}: {path.name}\n{'=' * 65}")

    if not path.exists():
        print("STOP: file not found")
        return

    header = pd.read_csv(path, nrows=0).columns.tolist()
    print("Columns:", header)

    def choose(candidates):
        return next((c for c in candidates if c in header), None)

    instance_col = choose(["scenario_instance_id", "instance_id"])
    cell_col = choose(["cell_id"])
    time_col = choose(["time_s"])
    soc_col = choose(["soc_proxy", "soc"])

    if not all([instance_col, cell_col, time_col, soc_col]):
        print("STOP: could not identify all required columns.")
        print("Detected:", instance_col, cell_col, time_col, soc_col)
        return

    usecols = list(dict.fromkeys([
        instance_col, cell_col, time_col, soc_col
    ]))

    rows = 0
    soc_min, soc_max = np.inf, -np.inf
    invalid_soc = 0
    duplicate_keys = 0
    invalid_examples = []
    duplicate_examples = []
    previous_last_key = None
    boundary_duplicate_count = 0

    for chunk in pd.read_csv(
        path, usecols=usecols, chunksize=CHUNK_SIZE
    ):
        rows += len(chunk)

        soc = pd.to_numeric(chunk[soc_col], errors="coerce")
        finite = soc[np.isfinite(soc)]

        if len(finite):
            soc_min = min(soc_min, float(finite.min()))
            soc_max = max(soc_max, float(finite.max()))

        bad = soc.isna() | ~np.isfinite(soc) | (soc < 0) | (soc > 1)
        invalid_soc += int(bad.sum())

        if len(invalid_examples) < 8 and bad.any():
            cols = [instance_col, cell_col, time_col, soc_col]
            invalid_examples.extend(
                chunk.loc[bad, cols].head(8 - len(invalid_examples))
                .to_dict("records")
            )

        keys = [instance_col, cell_col, time_col]
        duplicated = chunk.duplicated(keys, keep=False)
        duplicate_keys += int(duplicated.sum())

        if len(duplicate_examples) < 8 and duplicated.any():
            duplicate_examples.extend(
                chunk.loc[duplicated, keys].head(
                    8 - len(duplicate_examples)
                ).to_dict("records")
            )

        # Detect a repeated key exactly across adjacent chunk boundaries.
        first_key = tuple(chunk.iloc[0][keys]) if len(chunk) else None
        last_key = tuple(chunk.iloc[-1][keys]) if len(chunk) else None

        if first_key is not None and first_key == previous_last_key:
            boundary_duplicate_count += 1

        previous_last_key = last_key

    print("Rows scanned:", rows)
    print("SOC column:", soc_col)
    print("SOC min:", soc_min if np.isfinite(soc_min) else "no finite values")
    print("SOC max:", soc_max if np.isfinite(soc_max) else "no finite values")
    print("Rows with missing/non-finite/out-of-range SOC:", invalid_soc)
    print("Rows participating in duplicate keys within chunks:", duplicate_keys)
    print("Matching keys across adjacent chunk boundaries:", boundary_duplicate_count)
    print("Example invalid SOC rows:", invalid_examples)
    print("Example duplicate keys:", duplicate_examples)

for label, path in FILES.items():
    audit_csv(label, path)

print("\nAUDIT COMPLETE — no files were modified.")


raw: cell_timeseries.csv
Columns: ['scenario_id', 'scenario_instance_id', 'data_split', 'pack_topology', 'cell_id', 'state_kind', 'ageing_cycles', 'ambient_temperature_C', 'requested_charge_rate_C', 'conditioning_duration_s', 'time_s', 'cell_voltage_V', 'cell_current_A', 'cell_temperature_K', 'surface_temperature_K', 'ambient_temperature_K', 'soc_proxy', 'negative_particle_concentration_mean_mol_m3', 'positive_particle_concentration_mean_mol_m3', 'electrolyte_concentration_xavg_mol_m3', 'capacity_Ah_reference_test', 'capacity_retention_soh', 'dc_pulse_resistance_proxy_ohm', 'resistance_proxy_multiplier', 'pybamm_resistance_proxy_ohm', 'neg_sei_thickness', 'neg_crack_length', 'lam_negative']
Rows scanned: 7656014
SOC column: soc_proxy
SOC min: -0.05
SOC max: 0.2112651347153865
Rows with missing/non-finite/out-of-range SOC: 5626627
Rows participating in duplicate keys within chunks: 2038
Matching keys across adjacent chunk boundaries: 0
Example invalid SOC rows: [{'scenario_instance_id'

In [6]:
from pathlib import Path
import pandas as pd
import numpy as np

V1 = Path("/content/drive/MyDrive/PINN_EV_Battery/data/v1")
FILE = V1 / "canonical" / "cell_timeseries_canonical.csv"

cols = [
    "scenario_instance_id",
    "cell_id",
    "time_s",
    "cell_current_A",
    "cell_voltage_V",
    "soc_proxy",
]

stats = {}
sample = []
chunk_size = 200_000

for chunk in pd.read_csv(FILE, usecols=cols, chunksize=chunk_size):
    chunk["soc_proxy"] = pd.to_numeric(
        chunk["soc_proxy"], errors="coerce"
    )

    # Aggregate by scenario instance without retaining the full dataset.
    for instance, group in chunk.groupby("scenario_instance_id"):
        s = group["soc_proxy"].to_numpy(dtype=float)
        finite = s[np.isfinite(s)]

        if instance not in stats:
            stats[instance] = {
                "rows": 0,
                "finite": 0,
                "invalid": 0,
                "below_0": 0,
                "above_1": 0,
                "min": np.inf,
                "max": -np.inf,
                "sum": 0.0,
            }

        d = stats[instance]
        d["rows"] += len(group)
        d["finite"] += len(finite)
        d["invalid"] += int((~np.isfinite(s)).sum())
        d["below_0"] += int((s < 0).sum())
        d["above_1"] += int((s > 1).sum())

        if len(finite):
            d["min"] = min(d["min"], float(finite.min()))
            d["max"] = max(d["max"], float(finite.max()))
            d["sum"] += float(finite.sum())

    # Capture one trajectory for detailed inspection.
    mask = (
        (chunk["scenario_instance_id"] == "S1_HEALTHY")
        & (chunk["cell_id"] == 1)
    )
    if mask.any():
        sample.append(chunk.loc[mask].copy())

summary = []
for instance, d in sorted(stats.items()):
    summary.append({
        "instance": instance,
        "rows": d["rows"],
        "finite_soc": d["finite"],
        "invalid_soc": d["invalid"],
        "soc_below_0": d["below_0"],
        "soc_above_1": d["above_1"],
        "soc_min": d["min"],
        "soc_max": d["max"],
        "soc_mean": (
            d["sum"] / d["finite"] if d["finite"] else np.nan
        ),
    })

print("=== SOC SUMMARY BY INSTANCE ===")
display(pd.DataFrame(summary).round(6))

if sample:
    trajectory = pd.concat(sample, ignore_index=True)
    trajectory = trajectory.sort_values("time_s")

    print("\n=== S1_HEALTHY / CELL 1: FIRST 12 ROWS ===")
    display(trajectory[cols].head(12).round(6))

    print("\n=== S1_HEALTHY / CELL 1: LAST 12 ROWS ===")
    display(trajectory[cols].tail(12).round(6))

    print("\n=== S1_HEALTHY / CELL 1: SOC QUANTILES ===")
    display(
        trajectory["soc_proxy"]
        .describe(percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99])
        .to_frame("soc_proxy")
        .round(6)
    )

print("\nREAD-ONLY DIAGNOSTIC COMPLETE — no files were modified.")

=== SOC SUMMARY BY INSTANCE ===


,instance,rows,finite_soc,invalid_soc,soc_below_0,soc_above_1,soc_min,soc_max,soc_mean
0,S1_HEALTHY,269000,269000,0,199100,0,-0.05,0.200000,-0.002375
1,S2_HEALTHY,272700,272700,0,198300,0,-0.05,0.200000,0.000089
2,S3_HEALTHY,269500,269500,0,198300,0,-0.05,0.200000,-0.001255
3,S4_CELL004,269991,269991,0,199909,0,-0.05,0.209534,-0.002448
4,S4_CELL019,269991,269991,0,199909,0,-0.05,0.209534,-0.002448
5,S4_CELL037_BENCH,269991,269991,0,199909,0,-0.05,0.209534,-0.002448
6,S4_CELL055,269991,269991,0,199909,0,-0.05,0.209534,-0.002448
7,S4_CELL082,269991,269991,0,199909,0,-0.05,0.209534,-0.002448
8,S5_CELL004,275096,275096,0,200542,0,-0.05,0.211265,0.000170
9,S5_CELL004_082,276092,276092,0,201384,0,-0.05,0.211265,0.000069



=== S1_HEALTHY / CELL 1: FIRST 12 ROWS ===


,scenario_instance_id,cell_id,time_s,cell_current_A,cell_voltage_V,soc_proxy
0,S1_HEALTHY,1,0.000000,0.0,4.148384,0.2
1,S1_HEALTHY,1,0.000001,0.0,4.148411,0.2
2,S1_HEALTHY,1,0.000002,0.0,4.148438,0.2
3,S1_HEALTHY,1,0.000003,0.0,4.148492,0.2
4,S1_HEALTHY,1,0.000006,0.0,4.148599,0.2
5,S1_HEALTHY,1,0.000012,0.0,4.148810,0.2
6,S1_HEALTHY,1,0.000024,0.0,4.149223,0.2
7,S1_HEALTHY,1,0.000036,0.0,4.149626,0.2
8,S1_HEALTHY,1,0.000049,0.0,4.150018,0.2
9,S1_HEALTHY,1,0.000073,0.0,4.150777,0.2



=== S1_HEALTHY / CELL 1: LAST 12 ROWS ===


,scenario_instance_id,cell_id,time_s,cell_current_A,cell_voltage_V,soc_proxy
2678,S1_HEALTHY,1,7182.411162,0.0,4.195884,0.198079
2679,S1_HEALTHY,1,7184.585628,0.0,4.195849,0.198079
2680,S1_HEALTHY,1,7188.934560,0.0,4.195789,0.198079
2681,S1_HEALTHY,1,7192.848599,0.0,4.195742,0.198079
2682,S1_HEALTHY,1,7197.167863,0.0,4.195698,0.198079
2683,S1_HEALTHY,1,7201.487127,0.0,4.195660,0.198079
2684,S1_HEALTHY,1,7205.806391,0.0,4.195627,0.198079
2685,S1_HEALTHY,1,7210.125654,0.0,4.195599,0.198079
2686,S1_HEALTHY,1,7214.444918,0.0,4.195574,0.198079
2687,S1_HEALTHY,1,7218.764182,0.0,4.195553,0.198079



=== S1_HEALTHY / CELL 1: SOC QUANTILES ===


,soc_proxy
count,2690.000000
mean,-0.002375
std,0.084813
min,-0.050000
1%,-0.050000
5%,-0.050000
25%,-0.050000
50%,-0.050000
75%,0.008708
95%,0.198079



READ-ONLY DIAGNOSTIC COMPLETE — no files were modified.


# STOP HERE

If the final cell reports `STATUS: PASS`, the dataset is ready for the model-development stage.

## Next notebook

**Notebook 06 — V1 PINN Training & Model Validation**

That is the first notebook where we actually train the new V1 model.

It will include the B1 data-only baseline, the B3 physics-informed predictive model, strict trajectory-level splits, future-state prediction, checkpointing, physics residual evaluation, and ablation experiments.

### GPU

- Notebook 05: **CPU**
- Notebook 06: **Google Colab T4 GPU recommended**
- Do not build the frontend yet.

The 3D battery UI should be built only after the predictive model and localization/early-warning outputs have been experimentally validated.
